# Khel AI V2 — Ball-by-Ball Batting Form Model

This Google Colab notebook trains a first-version model for **batting form quantification** using the synthetic Khel AI cricket dataset from GitHub.

The notebook will:

1. Load ball-by-ball cricket data from the GitHub repository.
2. Create a `ball_impact_score` for each legal delivery faced by a batter.
3. Create a rolling `batting_form_index` between `0` and `1`.
4. Train a machine learning model to predict batting form.
5. Save the trained model as a `.pkl` file.
6. Export `batting_form_ball_by_ball.csv` and `player_form_index.csv`.

> Form scale:
>
> - `0.00 - 0.30`: struggling
> - `0.30 - 0.50`: below average
> - `0.50 - 0.70`: stable
> - `0.70 - 0.85`: good
> - `0.85 - 1.00`: excellent

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 2. Load dataset from GitHub

This notebook loads the dataset directly from:

`https://github.com/HimanshuKhale/datasets_cricket`

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/HimanshuKhale/datasets_cricket/main/"

ball_events = pd.read_csv(BASE_URL + "ball_events.csv")
ball_features = pd.read_csv(BASE_URL + "ball_training_features.csv")
player_features = pd.read_csv(BASE_URL + "player_match_features.csv")
players = pd.read_csv(BASE_URL + "players.csv")
innings = pd.read_csv(BASE_URL + "innings.csv")
matches = pd.read_csv(BASE_URL + "matches.csv")

print("ball_events:", ball_events.shape)
print("ball_features:", ball_features.shape)
print("player_features:", player_features.shape)
print("players:", players.shape)
print("innings:", innings.shape)
print("matches:", matches.shape)

ball_features.head()

## 3. Inspect columns

Run this cell once to understand the available fields.

In [ ]:
print("ball_events columns:")
print(ball_events.columns.tolist())

print("
ball_training_features columns:")
print(ball_features.columns.tolist())

print("
players columns:")
print(players.columns.tolist())

## 4. Merge ball events with training features

`ball_training_features.csv` contains numerical ML features.  
`ball_events.csv` contains shot metadata like shot type and shot zone.

In [ ]:
df = ball_features.merge(
    ball_events[[
        "id", "shot_type", "shot_zone", "wicket_fell", "wicket_type", "dismissed_player_id", "fielder_name", "notes"
    ]],
    left_on="ball_event_id",
    right_on="id",
    how="left"
)

df = df.drop(columns=["id"])

print(df.shape)
df.head()

## 5. Clean and prepare ball-level data

For V1, we calculate batting form only from **legal deliveries** because wides are not true batting actions.

In [ ]:
# Keep legal balls only for batting-form movement.
df = df[df["is_legal_delivery"] == True].copy()

# Safe missing-value handling.
df["required_run_rate"] = df["required_run_rate"].fillna(0)
df["pressure_index"] = df["pressure_index"].fillna(0)
df["batter_control_score"] = df["batter_control_score"].fillna(0.5)
df["shot_risk_score"] = df["shot_risk_score"].fillna(0.5)
df["expected_runs"] = df["expected_runs"].fillna(df["expected_runs"].median())
df["expected_wicket_probability"] = df["expected_wicket_probability"].fillna(0)
df["extras"] = df["extras"].fillna(0)
df["target_is_wicket"] = df["target_is_wicket"].fillna(0).astype(int)

# Ensure numeric columns are numeric.
numeric_cols = [
    "runs_off_bat", "extras", "current_run_rate", "required_run_rate", "pressure_index",
    "expected_runs", "expected_wicket_probability", "shot_risk_score", "batter_control_score",
    "target_is_wicket", "target_total_runs"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["runs_off_bat", "current_run_rate", "pressure_index", "batter_control_score", "shot_risk_score"])

print(df.shape)
df.head()

## 6. Create ball impact score

This is the cricket logic layer.

A ball can improve or reduce form depending on:

- Runs scored
- Batter control
- Shot risk
- Pressure response
- Whether the batter survived or got out

In [ ]:
def normalize_runs(runs):
    """
    Convert runs into a 0-1 score.
    Dot = low score, 1/2 = moderate, 4/6 = high.
    """
    if runs == 0:
        return 0.15
    elif runs == 1:
        return 0.45
    elif runs == 2:
        return 0.60
    elif runs == 3:
        return 0.70
    elif runs == 4:
        return 0.90
    elif runs >= 6:
        return 1.00
    return 0.30


def shot_quality_score(row):
    """
    Rewards controlled low-risk shots.
    Penalizes risky shots with poor control.
    """
    control = row["batter_control_score"]
    risk = row["shot_risk_score"]
    score = (0.65 * control) + (0.35 * (1 - risk))
    return np.clip(score, 0, 1)


def pressure_response_score(row):
    """
    Rewards good batting response under pressure.
    """
    pressure = row["pressure_index"]
    runs_score = normalize_runs(row["runs_off_bat"])
    control = row["batter_control_score"]

    response = (0.5 * runs_score) + (0.5 * control)

    if pressure >= 0.65:
        response = response * 1.10
    elif pressure <= 0.25:
        response = response * 0.95

    return np.clip(response, 0, 1)


def wicket_survival_score(row):
    """
    Wicket ball gives a major negative effect.
    Non-wicket ball gives survival credit.
    """
    if row["target_is_wicket"] == 1:
        return 0.0
    return 1.0


def calculate_ball_impact(row):
    run_score = normalize_runs(row["runs_off_bat"])
    control_score = row["batter_control_score"]
    shot_quality = shot_quality_score(row)
    pressure_response = pressure_response_score(row)
    survival = wicket_survival_score(row)

    impact = (
        0.30 * run_score +
        0.25 * control_score +
        0.20 * shot_quality +
        0.15 * pressure_response +
        0.10 * survival
    )

    # Extra wicket penalty.
    if row["target_is_wicket"] == 1:
        impact *= 0.35

    return np.clip(impact, 0, 1)


df["ball_impact_score"] = df.apply(calculate_ball_impact, axis=1)

df[[
    "ball_event_id", "innings_id", "over_number", "ball_number", "striker_id",
    "runs_off_bat", "pressure_index", "batter_control_score", "shot_risk_score",
    "target_is_wicket", "ball_impact_score"
]].head(20)

## 7. Create rolling batting form index

The batting form should not jump randomly. It should move smoothly ball by ball.

Formula:

```text
new_form = memory * previous_form + (1 - memory) * ball_impact
```

Default:

```text
memory = 0.85
starting_form = 0.50
```

In [ ]:
df = df.sort_values(["innings_id", "over_number", "ball_number", "ball_event_id"]).copy()


def calculate_batter_form(group, starting_form=0.50, memory=0.85):
    form_values = []
    current_form = starting_form

    for _, row in group.iterrows():
        current_form = memory * current_form + (1 - memory) * row["ball_impact_score"]
        current_form = np.clip(current_form, 0, 1)
        form_values.append(current_form)

    return pd.Series(form_values, index=group.index)


# This tracks each batter across the dataset in ball order.
df["batting_form_index"] = df.groupby("striker_id", group_keys=False).apply(
    lambda g: calculate_batter_form(g)
)

# A more local innings-level form is also useful for live-match dashboards.
df["innings_batting_form_index"] = df.groupby(["innings_id", "striker_id"], group_keys=False).apply(
    lambda g: calculate_batter_form(g)
)


df[[
    "ball_event_id", "innings_id", "striker_id", "runs_off_bat", "target_is_wicket",
    "ball_impact_score", "batting_form_index", "innings_batting_form_index"
]].head(30)

## 8. Add player names

This cell attempts to identify the player name column automatically.

In [ ]:
player_id_col = "id" if "id" in players.columns else players.columns[0]

possible_name_cols = ["name", "player_name", "full_name", "display_name"]
player_name_col = None
for col in possible_name_cols:
    if col in players.columns:
        player_name_col = col
        break

if player_name_col is None:
    # Fallback: use the second column if available.
    player_name_col = players.columns[1] if len(players.columns) > 1 else player_id_col

player_lookup = players[[player_id_col, player_name_col]].drop_duplicates()
player_lookup = player_lookup.rename(columns={player_id_col: "striker_id", player_name_col: "batter_name"})

df = df.merge(player_lookup, on="striker_id", how="left")

df[["striker_id", "batter_name", "runs_off_bat", "ball_impact_score", "batting_form_index"]].head()

## 9. Visualize one batter's form movement

In [ ]:
sample_batter = df["striker_id"].value_counts().index[0]
batter_df = df[df["striker_id"] == sample_batter].copy().reset_index(drop=True)
batter_df["ball_sequence"] = batter_df.index + 1

batter_name = batter_df["batter_name"].iloc[0] if "batter_name" in batter_df.columns else str(sample_batter)

plt.figure(figsize=(14, 5))
plt.plot(batter_df["ball_sequence"], batter_df["batting_form_index"], marker="o")
plt.title(f"Ball-by-ball Batting Form Index — {batter_name} / ID {sample_batter}")
plt.xlabel("Balls Faced")
plt.ylabel("Batting Form Index")
plt.ylim(0, 1)
plt.grid(True)
plt.show()

batter_df[["ball_sequence", "runs_off_bat", "target_is_wicket", "ball_impact_score", "batting_form_index"]].head(25)

## 10. Train batting form prediction model

This trains a model to predict the calculated `batting_form_index` from ball context features.

In [ ]:
features = [
    "runs_off_bat",
    "extras",
    "current_run_rate",
    "required_run_rate",
    "pressure_index",
    "expected_runs",
    "expected_wicket_probability",
    "shot_risk_score",
    "batter_control_score",
    "target_is_wicket",
    "target_total_runs"
]

target = "batting_form_index"

model_df = df[features + [target]].dropna().copy()

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

models = {
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        random_state=42
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results.append({"model": name, "MAE": mae, "R2": r2})
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values("MAE")
results_df

## 11. Select best model and inspect feature importance

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": features,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

    display(importance_df)

    plt.figure(figsize=(10, 5))
    plt.bar(importance_df["feature"], importance_df["importance"])
    plt.title("Feature Importance for Batting Form Model")
    plt.xlabel("Feature")
    plt.ylabel("Importance")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("This model does not expose feature_importances_.")

## 12. Save model package as PKL

The package includes:

- model
- feature columns
- target name
- model metrics
- scale explanation

In [ ]:
metrics = results_df.to_dict(orient="records")

model_package = {
    "model": best_model,
    "model_name": best_model_name,
    "feature_columns": features,
    "target": target,
    "metrics": metrics,
    "description": "Khel AI V2 ball-by-ball batting form model",
    "scale": "0 to 1 normalized batting form index",
    "form_scale_interpretation": {
        "0.00-0.30": "struggling",
        "0.30-0.50": "below average",
        "0.50-0.70": "stable",
        "0.70-0.85": "good",
        "0.85-1.00": "excellent"
    }
}

joblib.dump(model_package, "khel_ai_batting_form_model.pkl")
print("Saved: khel_ai_batting_form_model.pkl")

## 13. Export ball-by-ball and player-level form files

In [ ]:
ball_output_cols = [
    "ball_event_id", "innings_id", "over_number", "ball_number", "striker_id", "batter_name",
    "bowler_id", "runs_off_bat", "extras", "phase", "current_run_rate", "required_run_rate",
    "pressure_index", "expected_runs", "expected_wicket_probability", "shot_risk_score",
    "batter_control_score", "target_is_wicket", "shot_type", "shot_zone",
    "ball_impact_score", "batting_form_index", "innings_batting_form_index"
]

existing_ball_output_cols = [col for col in ball_output_cols if col in df.columns]

batting_form_ball_by_ball = df[existing_ball_output_cols].copy()
batting_form_ball_by_ball.to_csv("batting_form_ball_by_ball.csv", index=False)

player_form_index = (
    df.groupby(["striker_id", "batter_name"], dropna=False)
    .agg(
        balls_faced=("ball_event_id", "count"),
        total_runs=("runs_off_bat", "sum"),
        avg_ball_impact=("ball_impact_score", "mean"),
        latest_form_index=("batting_form_index", "last"),
        best_form_index=("batting_form_index", "max"),
        lowest_form_index=("batting_form_index", "min"),
        wickets=("target_is_wicket", "sum")
    )
    .reset_index()
    .sort_values("latest_form_index", ascending=False)
)

player_form_index.to_csv("player_form_index.csv", index=False)

print("Saved: batting_form_ball_by_ball.csv")
print("Saved: player_form_index.csv")

player_form_index.head(20)

## 14. Test prediction for a new ball

Change the values in `new_ball` to simulate a live match ball.

In [ ]:
loaded = joblib.load("khel_ai_batting_form_model.pkl")
model = loaded["model"]
feature_columns = loaded["feature_columns"]

new_ball = pd.DataFrame([{
    "runs_off_bat": 4,
    "extras": 0,
    "current_run_rate": 8.5,
    "required_run_rate": 9.2,
    "pressure_index": 0.72,
    "expected_runs": 1.4,
    "expected_wicket_probability": 0.08,
    "shot_risk_score": 0.45,
    "batter_control_score": 0.82,
    "target_is_wicket": 0,
    "target_total_runs": 4
}])

predicted_form = model.predict(new_ball[feature_columns])[0]
print("Predicted batting form:", round(predicted_form, 3))

## 15. Download outputs from Colab

Run this cell in Google Colab to download the PKL and CSV files.

In [ ]:
try:
    from google.colab import files

    files.download("khel_ai_batting_form_model.pkl")
    files.download("batting_form_ball_by_ball.csv")
    files.download("player_form_index.csv")
except Exception as e:
    print("Download works only inside Google Colab.")
    print("Files are saved in the current notebook directory.")
    print(e)

# Next upgrades for Khel AI V2

After this V1 works, the next version should add:

1. Previous 3 / 5 / 10 innings rolling form.
2. Batter-vs-bowler matchup history.
3. Venue-adjusted form.
4. Phase-wise form: powerplay, middle overs, death overs.
5. Commentary-based timing/control once the semantic commentary layer is ready.
6. Strict anti-leakage logic: current ball form should not use future match information.